# 00 — Silver Layer: Longitudinal Standardization for HCP Classification

This notebook prepares the raw weekly HCP dataset for downstream modeling and analysis.  
The design assumes that the prediction unit is the HCP and that each HCP contributes a fixed 86-week history.

Outputs generated by this notebook:

- `data/silver_layer_longitudinal.parquet`
- `data/hcp_manifest.parquet`
- `data/reports/column_inventory.csv`
- `data/reports/data_quality_summary.csv`
- `data/reports/negative_value_audit.csv`
- `data/reports/hcp_level_target_distribution.csv`

Main principles:

1. preserve weekly structure
2. build HCP-level label metadata without leakage
3. validate temporal integrity
4. create dense categorical views for specialty, state and age band
5. prepare fold assignments at HCP level for future modeling


In [ ]:
from pathlib import Path
import warnings
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 1. Configuration


In [ ]:
DATA_PATH = Path("data.csv")
OUTPUT_DIR = Path("data")
REPORT_DIR = OUTPUT_DIR / "reports"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
RANDOM_STATE = 42
AUTO_CLIP_CLEARLY_NONNEGATIVE = False  # keep False unless a business rule explicitly confirms clipping


## 2. Data ingestion


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {DATA_PATH.resolve()}")

try:
    df = pd.read_csv(DATA_PATH, engine="pyarrow")
except Exception:
    df = pd.read_csv(DATA_PATH)

print(f"Raw shape: {df.shape}")
print(df.head(3))


In [ ]:
column_inventory = pd.DataFrame({
    "column": df.columns,
    "dtype_raw": df.dtypes.astype(str).values
})

column_inventory.to_csv(REPORT_DIR / "column_inventory.csv", index=False)
column_inventory


## 3. Canonical column groups

These groups are based on the uploaded dataset.  
The lists are used later for validation, aggregation and reporting.


In [ ]:
ID_COL = "NUEVO_ID"
TIME_COL = "WEEK_ID"
TARGET_COL = "ATSEG"

specialty_dummy_cols = ["SPEC_GE", "SPEC_GPFM", "SPEC_IM", "SPEC_NRP", "SPEC_OTHER_SPEC", "SPEC_PHA"]
state_dummy_cols = ["STATE_1", "STATE_2", "STATE_3", "STATE_4", "STATE_5", "STATE_6", "STATE_7", "STATE_8", "STS_OTHER_STS"]
age_dummy_cols = ["(1940, 1960]", "(1960, 1980]", "(1980, 2000]", "(2000, 2020]", "(2020, 2030]"]

calendar_cols = ["YEAR", "QTR", "YEAR_QTR"]

trx_cols = [c for c in df.columns if c.endswith("_TRX")]
nrx_cols = [c for c in df.columns if c.endswith("_NRX")]
nbrx_cols = [c for c in df.columns if c.endswith("_NBRX")]
claim_cols = [c for c in df.columns if c.startswith("N_CLM")]
marketing_cols = ["RTE", "SAMPLES", "COPAY", "DIRECTMAIL", "SPK", "DETAILS"]
rolling_cols = [c for c in df.columns if "_R" in c and c.endswith("SUM")]
gidx_cols = [c for c in df.columns if c.endswith("_GIDX")]

predictor_cols = [
    c for c in df.columns
    if c not in [ID_COL, TIME_COL, TARGET_COL]
]

dynamic_numeric_cols = [
    c for c in (trx_cols + nrx_cols + nbrx_cols + claim_cols + marketing_cols + rolling_cols + gidx_cols + calendar_cols)
    if c in df.columns
]

static_dummy_cols = [c for c in (specialty_dummy_cols + state_dummy_cols + age_dummy_cols) if c in df.columns]

print("Dynamic numeric columns:", len(dynamic_numeric_cols))
print("Static dummy columns:", len(static_dummy_cols))


## 4. Type standardization and HCP-level label propagation


In [ ]:
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
if df[TIME_COL].isna().any():
    raise ValueError("WEEK_ID contains invalid dates after parsing.")

df = df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)

# Preserve raw target and create HCP-level propagated target
df["ATSEG_RAW"] = df[TARGET_COL]

hcp_target_map = (
    df[[ID_COL, TARGET_COL]]
    .dropna(subset=[TARGET_COL])
    .drop_duplicates(subset=[ID_COL])
    .set_index(ID_COL)[TARGET_COL]
    .to_dict()
)

df["ATSEG_HCP"] = df[ID_COL].map(hcp_target_map)
df["IS_LABELED_HCP"] = df["ATSEG_HCP"].notna().astype("int8")
df["WEEK_IDX"] = df.groupby(ID_COL).cumcount().astype("int16")

# Downcast numerics where appropriate
float64_cols = df.select_dtypes(include=["float64"]).columns.tolist()
int64_cols = df.select_dtypes(include=["int64"]).columns.tolist()

for col in float64_cols:
    df[col] = pd.to_numeric(df[col], downcast="float")

for col in int64_cols:
    if col in [ID_COL, "YEAR_QTR"]:
        continue
    df[col] = pd.to_numeric(df[col], downcast="integer")

print(df[[ID_COL, TIME_COL, TARGET_COL, "ATSEG_HCP", "IS_LABELED_HCP", "WEEK_IDX"]].head())


## 5. Dense categorical views from one-hot groups

The original dummy columns are preserved.  
Dense views are added for readability and downstream EDA.


In [ ]:
def recover_category_from_dummies(dataframe, columns, output_col, mapping=None):
    cols = [c for c in columns if c in dataframe.columns]
    if not cols:
        dataframe[output_col] = pd.Series([np.nan] * len(dataframe), index=dataframe.index, dtype="object")
        return dataframe

    tmp = dataframe[cols].copy()
    row_sum = tmp.sum(axis=1)
    winner = tmp.idxmax(axis=1)

    if mapping is not None:
        winner = winner.map(mapping)

    dataframe[output_col] = np.where(row_sum > 0, winner, np.nan)
    dataframe[output_col] = pd.Series(dataframe[output_col], index=dataframe.index, dtype="object")
    return dataframe

specialty_mapping = {
    "SPEC_GE": "GE",
    "SPEC_GPFM": "GPFM",
    "SPEC_IM": "IM",
    "SPEC_NRP": "NRP",
    "SPEC_OTHER_SPEC": "OTHER_SPEC",
    "SPEC_PHA": "PHA",
}

state_mapping = {
    "STATE_1": "STATE_1",
    "STATE_2": "STATE_2",
    "STATE_3": "STATE_3",
    "STATE_4": "STATE_4",
    "STATE_5": "STATE_5",
    "STATE_6": "STATE_6",
    "STATE_7": "STATE_7",
    "STATE_8": "STATE_8",
    "STS_OTHER_STS": "OTHER_STATE",
}

age_mapping = {
    "(1940, 1960]": "(1940, 1960]",
    "(1960, 1980]": "(1960, 1980]",
    "(1980, 2000]": "(1980, 2000]",
    "(2000, 2020]": "(2000, 2020]",
    "(2020, 2030]": "(2020, 2030]",
}

df = recover_category_from_dummies(df, specialty_dummy_cols, "SPECIALTY_GROUP", specialty_mapping)
df = recover_category_from_dummies(df, state_dummy_cols, "STATE_GROUP", state_mapping)
df = recover_category_from_dummies(df, age_dummy_cols, "AGE_RANGE_GROUP", age_mapping)

df[["SPECIALTY_GROUP", "STATE_GROUP", "AGE_RANGE_GROUP"]].head()


## 6. Data quality audit


In [ ]:
data_quality_summary = []

data_quality_summary.append({
    "check": "raw_rows",
    "value": int(df.shape[0])
})
data_quality_summary.append({
    "check": "raw_columns",
    "value": int(df.shape[1])
})
data_quality_summary.append({
    "check": "unique_hcps",
    "value": int(df[ID_COL].nunique())
})
data_quality_summary.append({
    "check": "unique_weeks",
    "value": int(df[TIME_COL].nunique())
})
data_quality_summary.append({
    "check": "feature_missing_rate_max_excluding_target",
    "value": float(df.drop(columns=[TARGET_COL, "ATSEG_RAW", "ATSEG_HCP"], errors="ignore").isna().mean().max())
})
data_quality_summary.append({
    "check": "target_missing_rate_row_level",
    "value": float(df[TARGET_COL].isna().mean())
})

duplicates = df.duplicated([ID_COL, TIME_COL]).sum()
data_quality_summary.append({
    "check": "duplicate_hcp_week_pairs",
    "value": int(duplicates)
})

rows_per_hcp = df.groupby(ID_COL).size()
weeks_per_hcp = df.groupby(ID_COL)[TIME_COL].nunique()

data_quality_summary.append({
    "check": "rows_per_hcp_min",
    "value": int(rows_per_hcp.min())
})
data_quality_summary.append({
    "check": "rows_per_hcp_median",
    "value": float(rows_per_hcp.median())
})
data_quality_summary.append({
    "check": "rows_per_hcp_max",
    "value": int(rows_per_hcp.max())
})
data_quality_summary.append({
    "check": "weeks_per_hcp_min",
    "value": int(weeks_per_hcp.min())
})
data_quality_summary.append({
    "check": "weeks_per_hcp_median",
    "value": float(weeks_per_hcp.median())
})
data_quality_summary.append({
    "check": "weeks_per_hcp_max",
    "value": int(weeks_per_hcp.max())
})

data_quality_summary = pd.DataFrame(data_quality_summary)
data_quality_summary.to_csv(REPORT_DIR / "data_quality_summary.csv", index=False)
data_quality_summary


In [ ]:
# Label consistency at HCP level
hcp_label_consistency = (
    df.groupby(ID_COL)["ATSEG_HCP"]
    .nunique(dropna=True)
    .value_counts(dropna=False)
    .rename_axis("n_unique_labels")
    .reset_index(name="hcp_count")
)

print(hcp_label_consistency)

# Weekly continuity audit
date_gap_audit = (
    df.groupby(ID_COL)[TIME_COL]
    .diff()
    .dropna()
    .dt.days
    .value_counts()
    .sort_index()
    .rename_axis("day_gap")
    .reset_index(name="count")
)

date_gap_audit.head(10)


## 7. Negative value audit

The dataset contains some index variables (`*_GIDX`) that can legitimately take negative values, so no blanket clipping is performed.

Only clearly non-negative operational variables are audited below.  
Examples: TRX, NRX, NBRX, claim counts, rolling sums, and promotional contact volumes.


In [ ]:
clearly_nonnegative_cols = [
    c for c in (trx_cols + nrx_cols + nbrx_cols + claim_cols + marketing_cols + rolling_cols)
    if c in df.columns
]

negative_value_audit = pd.DataFrame({
    "column": clearly_nonnegative_cols,
    "negative_count": [int((df[c] < 0).sum()) for c in clearly_nonnegative_cols],
    "minimum_value": [float(df[c].min()) for c in clearly_nonnegative_cols]
}).sort_values(["negative_count", "minimum_value"], ascending=[False, True])

negative_value_audit.to_csv(REPORT_DIR / "negative_value_audit.csv", index=False)
negative_value_audit.head(20)


In [ ]:
if AUTO_CLIP_CLEARLY_NONNEGATIVE:
    cols_to_clip = negative_value_audit.loc[negative_value_audit["negative_count"] > 0, "column"].tolist()
    for col in cols_to_clip:
        df[col] = df[col].clip(lower=0)
    print(f"Clipped {len(cols_to_clip)} clearly non-negative columns.")
else:
    print("AUTO_CLIP_CLEARLY_NONNEGATIVE = False. No clipping was applied.")


## 8. HCP manifest and fold assignment

Fold assignment is created at HCP level to preserve the correct prediction unit for future modeling.


In [ ]:
hcp_manifest = (
    df.groupby(ID_COL)
    .agg(
        n_rows=(ID_COL, "size"),
        n_weeks=(TIME_COL, "nunique"),
        first_week=(TIME_COL, "min"),
        last_week=(TIME_COL, "max"),
        ATSEG_HCP=("ATSEG_HCP", "first"),
        IS_LABELED_HCP=("IS_LABELED_HCP", "max"),
        SPECIALTY_GROUP=("SPECIALTY_GROUP", "first"),
        STATE_GROUP=("STATE_GROUP", "first"),
        AGE_RANGE_GROUP=("AGE_RANGE_GROUP", "first"),
    )
    .reset_index()
)

hcp_manifest["HCP_FOLD"] = -1

labeled_mask = hcp_manifest["IS_LABELED_HCP"] == 1
labeled_manifest = hcp_manifest.loc[labeled_mask].copy()

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
for fold, (_, valid_idx) in enumerate(skf.split(labeled_manifest[[ID_COL]], labeled_manifest["ATSEG_HCP"])):
    labeled_manifest.iloc[valid_idx, labeled_manifest.columns.get_loc("HCP_FOLD")] = fold

hcp_manifest.loc[labeled_mask, "HCP_FOLD"] = labeled_manifest["HCP_FOLD"].values

fold_map = dict(zip(hcp_manifest[ID_COL], hcp_manifest["HCP_FOLD"]))
df["HCP_FOLD"] = df[ID_COL].map(fold_map).fillna(-1).astype("int8")

hcp_target_distribution = (
    hcp_manifest["ATSEG_HCP"]
    .value_counts(dropna=False)
    .rename_axis("ATSEG_HCP")
    .reset_index(name="hcp_count")
)

hcp_target_distribution.to_csv(REPORT_DIR / "hcp_level_target_distribution.csv", index=False)

print(hcp_manifest.head())
print(hcp_target_distribution)


## 9. Persist curated assets


In [ ]:
silver_path = OUTPUT_DIR / "silver_layer_longitudinal.parquet"
manifest_path = OUTPUT_DIR / "hcp_manifest.parquet"

df.to_parquet(silver_path, index=False)
hcp_manifest.to_parquet(manifest_path, index=False)

print(f"Saved silver dataset to: {silver_path.resolve()}")
print(f"Saved HCP manifest to: {manifest_path.resolve()}")


## 10. Final review


In [ ]:
print("Silver dataset shape:", df.shape)
print("HCP manifest shape:", hcp_manifest.shape)
print(df[[ID_COL, TIME_COL, "WEEK_IDX", "ATSEG_HCP", "IS_LABELED_HCP", "HCP_FOLD", "SPECIALTY_GROUP", "STATE_GROUP", "AGE_RANGE_GROUP"]].head(10))
